# Rough Volatility (rBergomi)

Bayer, Friz, Gatheral (2016), *Pricing under rough volatility*.

This week is the conceptual break from W5 to W8. Every model in the toolkit so far
had a state that factorises through the present, which is exactly what gave us a
characteristic function and let us price with COS and Carr-Madan. rBergomi is the
first model that structurally refuses: variance is driven by a Volterra process with
a singular power-law kernel, so the state is path-dependent, non-Markovian, has no
closed-form characteristic function, and cannot be priced by COS. We return to Monte
Carlo, back to the W1/W4 discipline (report SE, z-score sanity, certify by
convergence rate not single points).

The payoff for that pain is the skew term structure. Roughness produces an ATM skew
that goes like a power law $\psi(\tau)\sim\tau^{H-1/2}$, which explodes into the short
end and persists into the long end with one exponent. Markovian models (Heston, Bates)
decay skew exponentially and cannot do both, which is precisely the flattening you
observed directly in real SPX at 30 vs 75 DTE.

## 1. Motivation: two empirical pillars

Roughness is supported by two independent bodies of evidence.

**Options side (skew power law).** The empirical ATM skew
$\psi(\tau)=|\partial_k\sigma_{\text{BS}}(k,\tau)|_{k=0}$ behaves like $\tau^{-\alpha}$
with $\alpha\approx0.4$, blowing up at short maturities. A stochastic-vol model
reproduces this only if its variance driver is rough with $H=\tfrac12-\alpha\approx0.1$.

**Time-series side (rough realized variance).** Gatheral, Jaisson, Rosenbaum (2018),
*Volatility is rough*, estimate $H$ directly from realized-variance series via the
scaling

$$\mathbb{E}\left|\log\sigma_{t+\delta}-\log\sigma_t\right|^{q}\sim\delta^{qH},$$

and find $H\approx0.1$ across thousands of assets, far rougher than any semimartingale
($H=\tfrac12$).

**Where the arc lands.** Local vol gets dynamics wrong; Heston fixes the level via
stochastic vol but is Markovian, so it flattens the skew term structure too fast;
jumps add short-dated skew a diffusion cannot; Bates combines them; and rough vol
fixes the *persistent term-structure* problem with a single roughness exponent. This
notebook is the last classical model of Phase 1, and its ML capstone (deep calibration)
is the first ML piece.

## 2. Fractional and Volterra Gaussian processes

**Fractional Brownian motion.** $B^H$ is the centred Gaussian process with

$$\mathbb{E}[B^H_s B^H_t]=\tfrac12\left(s^{2H}+t^{2H}-|t-s|^{2H}\right),\qquad H\in(0,1).$$

At $H=\tfrac12$ this is $\min(s,t)$, standard BM. It is self-similar
($B^H_{at}\overset{d}{=}a^H B^H_t$) with stationary increments

$$\mathbb{E}[(B^H_t-B^H_s)^2]=|t-s|^{2H}.$$

The increment autocovariance scales like $H(2H-1)k^{2H-2}$; for $H<\tfrac12$ that
prefactor is negative, so increments are anti-persistent (every move tends to reverse,
which is what makes a path look jagged). Paths are almost surely Hölder-$\gamma$ for
every $\gamma<H$ and no higher, so $H$ is the critical regularity exponent. Small $H$
is rough.

**The Volterra (Riemann-Liouville) process rBergomi uses.** Not fBm itself, but

$$\widetilde W_t=\sqrt{2H}\int_0^t(t-s)^{H-1/2}\,dW_s,\qquad
\text{Var}(\widetilde W_t)=2H\int_0^t(t-s)^{2H-1}\,ds=t^{2H}.$$

The $\sqrt{2H}$ normalises to the roughness scaling $t^{2H}$. The kernel exponent
$H-\tfrac12\in(-\tfrac12,0)$ for $H<\tfrac12$, so the kernel is *singular at $s=t$*:
it blows up (does not vanish) as $s\to t$, integrably because $H>0$.

Typical size of a move over horizon $\tau$ is the standard deviation,

$$\sqrt{\text{Var}(\widetilde W_\tau)}=\sqrt{\tau^{2H}}=\tau^{H}.$$

For $H<\tfrac12$, $\tau^{H}\gg\tau^{1/2}$ as $\tau\to0$, so the driver moves *much more*
than a diffusion over short horizons. This excess short-horizon motion is the engine
of the short-dated skew.

At $H=\tfrac12$ everything collapses to the familiar case: kernel $\equiv1$,
$\sqrt{2H}=1$, $\widetilde W_t=W_t$, and $v_t$ becomes a Markovian lognormal
(classical flat-Bergomi).

## 3. Why non-Markovian, and why COS is dead

$v_t$ integrates the *entire* history of $W$ against the kernel $(t-s)^{H-1/2}$,
weighted most heavily on the *recent* past (near the singularity at $s=t$) and decaying
into the distant past. Markov is not about recent-vs-distant weighting; it is about
whether the whole past compresses into the current value. It does not here, and the
reason is that the kernel does not factorise.

Contrast the OU / Heston kernel $e^{-\kappa(t-s)}$, which splits as
$e^{-\kappa\,dt}e^{-\kappa(t-s)}$, the $dt$ factor being $s$-free so it pulls out of
the integral:

$$Y_{t+dt}=\int_0^{t+dt}e^{-\kappa(t+dt-s)}\,dW_s
=e^{-\kappa\,dt}\,Y_t+\int_t^{t+dt}e^{-\kappa(t+dt-s)}\,dW_s.$$

The entire past collapses into one carried number $Y_t$. This *pull-out* is what makes
an exponential-kernel process Markov. A power law admits no identity
$(a+b)^p=f(a)\,g(b)^p$; the $dt$ stays welded inside the base, entangled with every
$s$. Advancing time re-scores the whole history against a new profile rather than
scaling a carried state, so you must keep the full path.

Consequence: $(X_t,v_t)$ is not a sufficient state, so there is no finite-dimensional
generator, no PDE, and no closed-form characteristic function. The affine-Riccati
machinery of W5 to W8 is gone. **Pricing is Monte Carlo.**

## 4. The rBergomi model

Under the pricing measure, with $r=0$ (forward measure; reinstate discounting
downstream),

$$v_t=\xi_0(t)\,\exp\left(\eta\,\widetilde W_t-\tfrac12\eta^2 t^{2H}\right),$$

$$\frac{dS_t}{S_t}=\sqrt{v_t}\,dW^S_t,\qquad
W^S=\rho\,W+\sqrt{1-\rho^2}\,W^\perp.$$

**Martingale correction, forced not chosen.** We want $\mathbb{E}[v_t]=\xi_0(t)$ so
that $\xi_0$ is genuinely the forward variance curve (the market input from variance
swaps / VIX). Since $\eta\widetilde W_t\sim\mathcal N(0,\eta^2 t^{2H})$ and
$\mathbb{E}[e^X]=e^{m+\sigma^2/2}$ for Gaussian $X$,

$$\mathbb{E}\left[e^{\eta\widetilde W_t}\right]=e^{\frac12\eta^2 t^{2H}}.$$

Multiplying by the reciprocal $e^{-\frac12\eta^2 t^{2H}}$ pins the mean to exactly $1$.
The $t^{2H}$ is $\text{Var}(\widetilde W_t)$; it re-pins itself automatically if $H$
changes. This is the same Doléans-Dade move as before, in plain lognormal form.

**Parameters and what each controls.**

| Parameter | Role | Acts on |
|---|---|---|
| $H\in(0,\tfrac12)$ | roughness (Hölder exponent) | the *exponent* of the skew term structure ($\tau^{H-1/2}$), i.e. its log-log slope |
| $\eta>0$ | vol-of-vol | the *amplitude* of the skew and smile |
| $\rho<0$ | price/vol correlation | the *sign and presence* of skew (no $\rho$, no tilt) |
| $\xi_0(\cdot)$ | forward variance curve | the *level and term structure of variance* (market input) |

$H$ rotates the log-log skew line; $\eta$ shifts it vertically. Rotation and shift are
independent, which is why calibration is well-posed and why $H,\eta$ do not degenerate
into each other. A pinned $\eta$ with interior $H$ is therefore a data or $\xi_0$
problem, not a roughness problem.

## 5. Short-time skew asymptotics

Power-counting heuristic (exponent robust, constant model-specific). Skew is
correlation times "vol motion measured against the price's own diffusive yardstick,"
each over horizon $\tau$:

$$\psi(\tau)\sim\rho\,\eta\cdot\frac{\text{vol driver move over }\tau}
{\text{price diffusive scale over }\tau}
\sim\rho\,\eta\cdot\frac{\tau^{H}}{\tau^{1/2}}=\rho\,\eta\,\tau^{\,H-1/2}.$$

Regimes from the exponent $H-\tfrac12$:

- $H=\tfrac12$: $\tau^0=$ const, skew flattens to a finite short-end value (the classical failure).
- $H<\tfrac12$: exponent negative, skew explodes as $\tau\to0$ (matches market; $H\approx0.1$ gives $\approx\tau^{-0.4}$).

At the long end the power law decays *polynomially*, whereas a Markovian model's
exponential-kernel autocovariance $e^{-\kappa\tau}$ decays *exponentially* and dies too
fast. One roughness exponent fixes both ends; this is the 30-vs-75 DTE flattening.

Two honesty notes. The heuristic gets the exponent right but the rigorous short-time
result is a theorem (Fukasawa; Bayer, Friz, Gatheral). And $\tau^{H-1/2}$ is a
short-$\tau$ statement; across the full surface $\xi_0$ and $\eta$ also shape things, so
do not expect a perfect single slope across all maturities. The log-log skew fit
(Demo 4) recovers a slope *near* $H-\tfrac12$, which is enough to see the roughness.

**Identifiability corollary.** Roughness is a multi-maturity phenomenon by
construction: one maturity is one point on the log-log line, and you cannot fit a slope
to one point. This is exactly why the single-slice W8 benchmark could not test it, and
why the deferred multi-maturity benchmark becomes essential here.

## 6. Simulation strategy (ground-truth-first)

No characteristic function, so we simulate. Build the **exact (Cholesky)** simulator
first as ground truth, then optionally the **hybrid scheme** (Bennedsen, Lunde,
Pakkanen 2017, $O(N\log N)$) as the fast production version, and check they agree.

We need the joint Gaussian of the Volterra values $\widetilde W_{t_i}$ and the driving
BM $W_{t_i}$, because the price uses $W^S=\rho W+\sqrt{1-\rho^2}W^\perp$ and the skew
comes from the price sharing $W$ with $\widetilde W$. Closed-form covariances on the
grid, for $s\le t$:

$$\mathbb{E}[\widetilde W_s\widetilde W_t]
=2H\int_0^s (s-u)^{H-1/2}(t-u)^{H-1/2}\,du
\qquad(\text{diagonal}=t^{2H}),$$

$$\mathbb{E}[W_s W_t]=\min(s,t),$$

$$\mathbb{E}[\widetilde W_t\, W_s]
=\frac{\sqrt{2H}}{H+\tfrac12}\left(t^{H+1/2}-(t-\min(s,t))^{H+1/2}\right).$$

The autocovariance integral has a closed form via ${}_2F_1$, or evaluate it by
quadrature; either is fine, and we verify the simulator against it regardless.
Assemble the $2N\times2N$ joint covariance $\Sigma$ from these three blocks,
Cholesky-factor once, and map an $(n_{\text{paths}},2N)$ iid standard-normal draw
through the factor.

**Implementation caution.** The $\Delta W$ increments used to build the price's $W^S$
must be the *same* Brownian increments that built $\widetilde W$; that shared $W$ is
the leverage channel. If you draw an independent BM for the price you will get zero
skew regardless of $\rho$.

## 7. Phase 1 ML capstone (stub, plan only)

**Deep calibration.** Train a neural network to approximate the map
$(H,\eta,\rho,\xi_0)\mapsto$ implied-vol surface, using the certified MC pricer above
as ground truth. This turns calibration from slow-MC-per-evaluation into a millisecond
forward pass, which is the industry use case that made rough vol practical.

Discipline (same as the whole project): the pricer is the ground truth, so **validate
the NN against the pricer, not the market**. Guiding principle: ML as a fast
approximator on a certified classical baseline, never a black-box predictor of a
low-signal quantity. Plan: fix a $\xi_0$ parameterisation, sample $(H,\eta,\rho)$ over
sensible ranges, generate surfaces with the MC pricer, train, then hold out and check
surface RMSE against fresh MC. Detailed design when we get here.